In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
import os

In [ ]:
print(load_dotenv(override=True))
current_dir = os.getcwd()

In [ ]:
pdf_path = os.path.join(current_dir, "Profile.pdf")
reader = PdfReader(pdf_path)
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin+=text

In [ ]:
with open("me.txt", 'r', encoding='UTF-8') as file:
    summary = file.read()

In [ ]:
system_prompt = f""" ALways answer within 10 words
# Your role
You are a digital twin running on a website chatting with visitors of the website
You represent the person whose website you are on.
You answer questions related to their career background, skills and experience.

Here are the details of the person you are representing:{summary}

If asked you explain clearly you are just an AI digital twin of the person

# Context
Here is the summary of the person's linkedin profile so that you can answer questions:
{linkedin}
Always stay in character as the digital twin of the person.
"""

display(Markdown(system_prompt))


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv('Gemini_API'))

messages = [{'role': 'system', 'content': system_prompt},
{'role': 'user', 'content': 'Hi tell me about yourself'}]

def convert_messages(messages):
    system_prompt = None
    contents = []

    for message in messages:
        role = message["role"]
        content = message["content"]

        # NEW
        if isinstance(content, list):
            content = " ".join(
                item["text"]
                for item in content
                if isinstance(item, dict) and "text" in item
            )

        if role == "system":
            system_prompt = content

        elif role == "assistant":
            contents.append(
                types.Content(
                    role="model",
                    parts=[types.Part(text=content)]
                )
            )

        else:
            contents.append(
                types.Content(
                    role="user",
                    parts=[types.Part(text=content)]
                )
            )

    return system_prompt, contents

def chat(message, history):
    history = [{'role':h["role"], "content":h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    system_instruction, contents = convert_messages(messages)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction
        )
    )

    return response.text

# chat("please tell me about yourself", [])

## Adding tools here without langchain

In [ ]:
def record_email(emails):

    if isinstance(emails, str):
        emails = [emails]

    with open("emails.txt", "a", encoding="utf-8") as f:
        for email in emails:
            f.write(email + "\n")

    return f"Recorded {len(emails)} emails"

In [ ]:
record_email("rohit@gmail.com")

In [ ]:
# Step 1 to tool calling write JSON to describe the tool:
# this part is automated by langchain this is one of the differences

from google.genai import types

tools = [
    types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="record_email_tool",
                description="Record a user's email",
                parameters={
                    "type": "OBJECT",
                    "properties": {
                        "email": {
                            "type": "STRING",
                            "description": "User email address"
                        }
                    },
                    "required": ["email"]
                }
            )
        ]
    )
]

In [ ]:
## Step 2 a new chat() function with tools incorporated
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    system_instruction, contents = convert_messages(messages)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools
        )
    )

    recorded = []

    for candidate in response.candidates:
        for part in candidate.content.parts:

            if part.function_call:
                tool_name = part.function_call.name
                args = dict(part.function_call.args)

                print("TOOL:", tool_name)
                print("ARGS:", args)

                if tool_name == "record_email_tool":
                    record_email(args["email"])
                    recorded.append(args["email"])

    if recorded:
        return f"Recorded {len(recorded)} emails: {', '.join(recorded)}"

    return response.text or "Action Completed"

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

In [ ]:
## Adding the push notification tool

In [ ]:
pushover_url = "https://api.pushover.net/1/messages.json"
# implementing pushover notifications
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
print(pushover_token[:4])

import requests
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

push("HEY!")

In [ ]:
def record_user_details(
    emails=None,
    names=None,
    notes="not provided"
):
    emails = emails or []
    names = names or []

    with open("contacts.txt", "a", encoding="utf-8") as f:
        f.write(
            f"Names: {names} | "
            f"Emails: {emails} | "
            f"Notes: {notes}\n"
        )

    push(
        f"Recording names={names}, "
        f"emails={emails}, "
        f"notes={notes}"
    )

    return (
        f"Recorded {len(names)} name(s) and "
        f"{len(emails)} email(s)"
    )

In [ ]:
def unknown_question(question):
    print("The given question could not be understood")
    return "OK"

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Record a user's contact details and notes",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The user's email address"
            },
            "name": {
                "type": "string",
                "description": "The user's name"
            },
            "notes": {
                "type": "string",
                "description": "Additional notes or interests"
            }
        },
        "required": ["email"]
    }
}

In [ ]:
unknown_question_json = {
    "name": "unknown_question",
    "description": "Use when the user's question cannot be answered from the available information",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Question that could not be understood or answered"
            }
        },
        "required": ["question"]
    }
}

In [ ]:
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    system_instruction, contents = convert_messages(messages)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools,
        )
    )

    if response.candidates:
        for candidate in response.candidates:
            if candidate.content and candidate.content.parts:
                for part in candidate.content.parts:
                    if part.function_call:
                        tool_name = part.function_call.name
                        args = dict(part.function_call.args)

                        print("TOOL:", tool_name)
                        print("ARGS:", args)

                        if tool_name == "record_user_details":
                            return record_user_details(
                                emails=args.get("emails", []),
                                names=args.get("names", []),
                                notes=args.get("notes", "not provided")
                            )

                        elif tool_name == "record_email_tool" or tool_name == "record_email":
                            email = args.get("email") or args.get("emails")
                            record_email(email)
                            return f"Recorded email: {email}"

                        elif tool_name == "unknown_question":
                            return unknown_question(args["question"])

    # Fallback to prevent returning None if the tool wasn't handled or model returned no text
    return response.text or "Action processed successfully."


In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

In [ ]:
def push(message):
    print(f"Sending Push Notification: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    try:
        requests.post(pushover_url, data=payload)
    except Exception as e:
        print("Push notification failed:", e)

def record_user_details(name="not provided", email="not provided", phone="not provided", notes="not provided"):
    #Save contact details to contacts.txt
    with open("contacts.txt", "a", encoding="utf-8") as f:
        f.write(f"Name: {name} | Email: {email} | Phone: {phone} | Notes: {notes}\n")

    # Send Push Notification
    notification_msg = (
        f"📱 New Contact Recorded!\n"
        f"Name: {name}\n"
        f"Email: {email}\n"
        f"Phone: {phone}\n"
        f"Notes: {notes}"
    )
    push(notification_msg)

    return f"Thank you! Recorded details for {name} ({email}, {phone}) and sent notification!"


In [ ]:
from google.genai import types

tools = [
    types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="record_user_details",
                description="Record a user's contact details (name, email, phone number, and optional notes) and send a push notification to the owner.",
                parameters={
                    "type": "OBJECT",
                    "properties": {
                        "name": {
                            "type": "STRING",
                            "description": "The full name of the user"
                        },
                        "email": {
                            "type": "STRING",
                            "description": "The user's email address"
                        },
                        "phone": {
                            "type": "STRING",
                            "description": "The user's phone number"
                        },
                        "notes": {
                            "type": "STRING",
                            "description": "Any additional context, message, or questions provided by the user"
                        }
                    },
                    "required": ["email"]  # requires at least email
                }
            )
        ]
    )
]


In [ ]:
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    system_instruction, contents = convert_messages(messages)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools,
        )
    )

    # Check for tool call
    if response.candidates:
        for candidate in response.candidates:
            if candidate.content and candidate.content.parts:
                for part in candidate.content.parts:
                    if part.function_call:
                        tool_name = part.function_call.name
                        args = dict(part.function_call.args)

                        print("TOOL CALLED:", tool_name)
                        print("ARGS:", args)

                        if tool_name == "record_user_details":
                            return record_user_details(
                                name=args.get("name", "not provided"),
                                email=args.get("email", "not provided"),
                                phone=args.get("phone", "not provided"),
                                notes=args.get("notes", "not provided")
                            )

    # Fallback to model text or default message (never returns None to Gradio)
    return response.text or "Your details have been saved!"


In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)